# 🎲 Recreating the "Game of Globes" (Outreach Activity)

This notebook demonstrates how to create the models for the "Game of Globes" outreach activity (Section 3.2.4 of Koelemeijer & Winterbourne 2021).

### 🌎 Scientific Context
When people ask, *"How do we know what is deep inside the Earth if we've never been there?"*, we use this game as a tactile analogy:
- We print multiple Earth globes that look identical from the outside, but each holds a different material sphere inside its hollow core (e.g., steel, glass, wood, a hollow ping-pong ball, or a mini Mars globe).
- By shaking and weighing the globes, the audience can guess what is inside based on weight (gravity) and sound (seismic wave reflections).

In this notebook, we configure a custom hollow cavity of exactly **20.4 mm diameter** (10.2 mm radius) to fit standard 20 mm hobby marbles or balls.

## Step 1: Import Libraries

In [ ]:
import os
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    calculate_displacement_scale
)

## Step 2: Build the Earth Shell with Custom Cavity Radius

For our 40 mm radius globe, we want an inner hollow radius of exactly **10.2 mm** (giving a tiny bit of clearance so the 20 mm insert ball can move and rumble freely inside). 

The required `inner_ratio` is $10.2 / 40.0 = 0.255$.

In [ ]:
model_radius_mm = 40.0
target_cavity_radius_mm = 10.2  # fits a standard 20mm ball
inner_ratio = target_cavity_radius_mm / model_radius_mm  # 0.255

# 1. Initialize the hollow model
model = GlobeModel(
    n_points=6000,
    radius=model_radius_mm,
    hollow=True,
    inner_ratio=inner_ratio
)

# 2. Load topography and displace outer surface
full_grid = GeographicGrid.from_netcdf(
    "../../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc", 'lat', 'lon', 'z'
)
grid_ds = GeographicGrid(
    lats=full_grid.lats[::15], lons=full_grid.lons[::15], grid=full_grid.grid[::15, ::15]
)
scale = calculate_displacement_scale(model_radius_mm, vertical_exagg=50.0, grid_units='m')
model.outer.displace(GridDisplacer(grid_ds), scale=scale)

print(f"Created globe with outer radius {model_radius_mm}mm and inner cavity radius {target_cavity_radius_mm}mm.")

## Step 3: Configure Magnet Joints & Export

We add magnet voids to hold the hemispheres together during the game, and export them.

In [ ]:
# Configure magnets (placed inside the thick 30mm shell wall, no bosses needed)
model.configure_magnets(
    diameter=5.0,
    height=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    n_magnets=3,
    add_bosses=False, # fit directly in the thick shell
)

os.makedirs('../../outputs', exist_ok=True)
model.export_hemispheres(
    "../../outputs/game_globe_top.stl",
    "../../outputs/game_globe_bottom.stl",
    engine='manifold'
)
print("Game of Globes hemispheres exported to outputs/")